# Notebook 4 — closing the three gaps in the manuscript

Three things the draft marks as open, in the order a referee would hit them.

**Gap 1 — the diagnostic and the main table cover different graphs.** Table 1 of the draft
reports $\alpha_H/\kappa$ on five hypergraphs; Table 2 reports speedups on ten. Here we
compute the diagnostic on all ten, exactly (max-flow), so the two tables line up.

**Gap 2 — one pattern only.** Everything so far is $K_3^{\times}$. We add the
cross-hyperedge four-cycle. This is not just "another pattern": since $\rho(C_4)=2$ and the
Fichtenberger–Peng decomposition of $C_4$ is a perfect matching of two edges, **both**
elementary draws are base-edge draws against the global normalizer. There is no
closing-vertex step, hence no localization term at all. Whatever gain appears is the
predictor's, with no localization confound — which makes $C_4$ the clean test of the
permutation reversal.

**Gap 3 — the `contact-high-school` anomaly.** It is the one graph where the learned
predictor is worse than no predictor (recovery $-0.076$, prediction correlation $0.282$).
The draft asserts an explanation; here we measure it, and check whether the same statistic
explains the spread across all ten graphs.

The last cell prints LaTeX rows ready to paste into the manuscript.

## 0. Library

In [ ]:
"""Paper-grade machinery for prediction-augmented counting of cross-hyperedge triangles.

Everything that is evaluated many times (many predictors, many permutation seeds) is
vectorised over precomputed triangle triples, so the expensive structural pass runs once.
"""
import numpy as np


def _bits_to_idx(x, nbytes):
    """Indices of the set bits of a Python int, fast."""
    if x == 0:
        return np.empty(0, dtype=np.int32)
    b = np.frombuffer(x.to_bytes(nbytes, 'little'), dtype=np.uint8)
    return np.flatnonzero(np.unpackbits(b, bitorder='little')).astype(np.int32)


def _popcount(x):
    try:
        return x.bit_count()
    except AttributeError:
        return bin(x).count('1')


class CrossHG:
    """Projected graph of a hypergraph + exact cross-triangle structure.

    Cross triangle: {u,v,w} pairwise adjacent in the projection, with no single
    hyperedge containing all three.  Canonical discovery follows the Fichtenberger-Peng
    path used by PredCount: order the three vertices by (degree, id) as x < y < z,
    base edge {x,y}, pivot x, closing vertex z.
    """

    def __init__(self, hyperedges, max_size=25, max_triples=4_000_000, seed=0, verbose=True):
        hs = [tuple(sorted(set(h))) for h in hyperedges]
        hs = [h for h in hs if 2 <= len(h) <= max_size]
        hs = list(dict.fromkeys(hs))
        verts = sorted({v for h in hs for v in h})
        idx = {v: i for i, v in enumerate(verts)}
        self.n = n = len(verts)
        self.nbytes = (n + 7) // 8
        self.hs = [tuple(idx[v] for v in h) for h in hs]
        self.hbits = [sum(1 << a for a in h) for h in self.hs]
        self.hsize = np.array([len(h) for h in self.hs], dtype=np.int32)

        memb = [[] for _ in range(n)]
        adj = [0] * n
        for hid, h in enumerate(self.hs):
            bits = self.hbits[hid]
            for a in h:
                memb[a].append(hid)
                adj[a] |= bits & ~(1 << a)
        self.memb = [set(m) for m in memb]
        self.adj = adj

        # ---- edges, CSR adjacency, edge ids
        nbr = [_bits_to_idx(adj[u], self.nbytes) for u in range(n)]
        self.deg = np.array([len(x) for x in nbr], dtype=np.int64)
        self.indptr = np.zeros(n + 1, dtype=np.int64)
        np.cumsum(self.deg, out=self.indptr[1:])
        self.indices = np.concatenate(nbr) if n else np.empty(0, np.int32)

        eu, ev = [], []
        for u in range(n):
            w = nbr[u][nbr[u] > u]
            eu.append(np.full(len(w), u, dtype=np.int32))
            ev.append(w)
        self.eu = np.concatenate(eu) if n else np.empty(0, np.int32)
        self.ev = np.concatenate(ev) if n else np.empty(0, np.int32)
        self.m = len(self.eu)
        self.eid = {}
        for i in range(self.m):
            self.eid[(int(self.eu[i]), int(self.ev[i]))] = i
        # edge id for every CSR slot
        self.slot_eid = np.empty(len(self.indices), dtype=np.int64)
        for u in range(n):
            a, b = self.indptr[u], self.indptr[u + 1]
            for k in range(a, b):
                v = int(self.indices[k])
                self.slot_eid[k] = self.eid[(u, v)] if u < v else self.eid[(v, u)]

        self._core_numbers()
        self.rank = np.empty(n, dtype=np.int64)
        order = np.lexsort((np.arange(n), self.deg))
        self.rank[order] = np.arange(n)

        self._cross_and_triples(max_triples, seed, verbose)
        self._features()

    # ------------------------------------------------------------------ structure
    def _core_numbers(self):
        n = self.n
        deg = self.deg.copy()
        core = np.zeros(n, dtype=np.int64)
        removed = np.zeros(n, dtype=bool)
        maxd = int(deg.max()) if n else 0
        buckets = [set() for _ in range(maxd + 1)]
        for v in range(n):
            buckets[deg[v]].add(v)
        k, i = 0, 0
        for _ in range(n):
            while i <= maxd and not buckets[i]:
                i += 1
            if i > maxd:
                break
            v = buckets[i].pop()
            k = max(k, i)
            core[v] = k
            removed[v] = True
            for j in range(self.indptr[v], self.indptr[v + 1]):
                w = int(self.indices[j])
                if removed[w]:
                    continue
                d = int(deg[w])
                buckets[d].discard(w)
                deg[w] = d - 1
                buckets[d - 1].add(w)
                if d - 1 < i:
                    i = d - 1
        self.core = core
        self.kappa = int(k)

    def _cross_and_triples(self, max_triples, seed, verbose):
        """Per-edge cross-triangle counts (exact) and the triple arrays (possibly sampled)."""
        nb = self.nbytes
        t = np.zeros(self.m, dtype=np.int64)
        covn = np.zeros(self.m, dtype=np.int64)     # covered common neighbours
        nhy = np.zeros(self.m, dtype=np.int64)      # hyperedges containing the edge
        hsum = np.zeros(self.m, dtype=np.int64)     # sum over those of (|h|-2)
        hmax = np.zeros(self.m, dtype=np.int64)
        covbits = [0] * self.m
        for i in range(self.m):
            u, v = int(self.eu[i]), int(self.ev[i])
            common = self.adj[u] & self.adj[v]
            S = self.memb[u] & self.memb[v]
            cov = 0
            for h in S:
                cov |= self.hbits[h]
            covbits[i] = cov
            nhy[i] = len(S)
            if S:
                sz = self.hsize[list(S)]
                hsum[i] = int((sz - 2).sum())
                hmax[i] = int(sz.max())
                covn[i] = _popcount(cov & common)
            t[i] = _popcount(common & ~cov)
        self.t = t
        self.covn, self.nhy, self.hsum, self.hmax = covn, nhy, hsum, hmax
        self.n_cross_total = int(t.sum()) // 3

        # triples: each cross triangle is generated once, from its canonical base edge
        target = self.n_cross_total if self.n_cross_total else 1
        p = min(1.0, max_triples / target)
        rng = np.random.default_rng(seed)
        keep = np.ones(self.m, dtype=bool) if p >= 1.0 else (rng.random(self.m) < p)
        self.triple_scale = 1.0 / p
        self.triple_fraction = p

        base, pivot, xz = [], [], []
        rank = self.rank
        for i in np.flatnonzero(keep):
            i = int(i)
            if t[i] == 0:
                continue
            u, v = int(self.eu[i]), int(self.ev[i])
            cross = (self.adj[u] & self.adj[v]) & ~covbits[i]
            z = _bits_to_idx(cross, nb)
            hi = max(rank[u], rank[v])
            z = z[rank[z] > hi]
            if not len(z):
                continue
            x, y = (u, v) if rank[u] < rank[v] else (v, u)
            base.append(np.full(len(z), i, dtype=np.int32))
            pivot.append(np.full(len(z), x, dtype=np.int32))
            xz.append(np.array([self.eid[(min(x, int(c)), max(x, int(c)))] for c in z],
                               dtype=np.int32))
        self.base = np.concatenate(base) if base else np.empty(0, np.int32)
        self.pivot = np.concatenate(pivot) if pivot else np.empty(0, np.int32)
        self.xz = np.concatenate(xz) if xz else np.empty(0, np.int32)
        if verbose:
            print(f'    n={self.n} m={self.m} kappa={self.kappa} '
                  f'#cross={self.n_cross_total} triples kept={len(self.base)} '
                  f'({100*self.triple_fraction:.1f}% of edges)')

    def _features(self):
        lg = np.log1p
        du, dv = self.deg[self.eu], self.deg[self.ev]
        cu, cv = self.core[self.eu], self.core[self.ev]
        ku = np.array([len(self.memb[u]) for u in self.eu], dtype=np.int64)
        kv = np.array([len(self.memb[v]) for v in self.ev], dtype=np.int64)
        A = np.column_stack([                       # Array-paper block: degrees + cores
            lg(np.minimum(du, dv)), lg(np.maximum(du, dv)),
            lg(du) + lg(dv), np.abs(lg(du) - lg(dv)),
            lg(np.minimum(cu, cv)), lg(np.maximum(cu, cv)), lg(cu) + lg(cv),
        ])
        B = np.column_stack([                       # cheap higher-order block
            lg(np.minimum(ku, kv)), lg(np.maximum(ku, kv)),
            lg(self.nhy), lg(self.hsum), lg(self.hmax),
        ])
        Cc = np.column_stack([lg(self.covn)])       # covered-neighbour block
        self.F = {'A': A, 'AB': np.hstack([A, B]), 'ABC': np.hstack([A, B, Cc])}
        self.y = np.log1p(self.t.astype(float))

    # ------------------------------------------------------------- success probability
    def _D(self, wp):
        """D_x = sum over neighbours c of (w({x,c})+1), as an array over vertices."""
        seg = wp[self.slot_eid]
        out = np.add.reduceat(seg, self.indptr[:-1])
        return out

    def p_succ(self, w, first_edge_only=False):
        wp = np.asarray(w, dtype=float) + 1.0
        W = wp.sum()
        D = self._D(wp)
        if first_edge_only:
            inner = 1.0 / self.deg[self.pivot]
        else:
            inner = wp[self.xz] / D[self.pivot]
        return float((wp[self.base] * inner).sum()) / W * self.triple_scale

    def p_base(self):
        return self.n_cross_total / (2 * self.m) ** 1.5

    # -------------------------------------------------------------------- predictors
    def w_perfect(self):
        return self.t.astype(float)

    def w_uniform(self):
        return np.zeros(self.m)

    def w_mindeg(self):
        return np.minimum(self.deg[self.eu], self.deg[self.ev]).astype(float)

    def w_permuted(self, w, seed):
        rng = np.random.default_rng(seed)
        return rng.permutation(np.asarray(w, dtype=float))


# ------------------------------------------------------------------------ ridge model

def ridge_fit(X, y, lam=1.0):
    mx, sx = X.mean(0), X.std(0) + 1e-12
    Z = (X - mx) / sx
    my, sy = y.mean(), y.std() + 1e-12
    yz = (y - my) / sy
    Z1 = np.hstack([Z, np.ones((len(Z), 1))])
    A = Z1.T @ Z1 + lam * np.eye(Z1.shape[1])
    A[-1, -1] -= lam                       # do not penalise the intercept
    beta = np.linalg.solve(A, Z1.T @ yz)
    return dict(beta=beta, mx=mx, sx=sx, my=my, sy=sy)


def ridge_predict_weights(model, X):
    Z = (X - model['mx']) / model['sx']
    Z1 = np.hstack([Z, np.ones((len(Z), 1))])
    yz = Z1 @ model['beta']
    y = yz * model['sy'] + model['my']
    return np.clip(np.expm1(np.clip(y, 0, 30)), 0, None)


# ------------------------------------------------------------------------- reporting

def evaluate(hg, n_perm=50, seed=0):
    """Localization, perfect predictor, permutation null, cheap heuristic, first-edge."""
    pb = hg.p_base()
    p_loc = hg.p_succ(hg.w_uniform())
    p_perf = hg.p_succ(hg.w_perfect())
    p_mind = hg.p_succ(hg.w_mindeg())
    p_first = hg.p_succ(hg.w_perfect(), first_edge_only=True)
    perms = np.array([hg.p_succ(hg.w_permuted(hg.w_perfect(), seed + b))
                      for b in range(n_perm)])
    s_perf, s_perm = p_perf / p_loc, perms / p_loc
    gain = s_perf - 1.0
    shares = (s_perf - s_perm) / gain if abs(gain) > 1e-12 else np.full(n_perm, np.nan)
    pval = (1 + (s_perm >= s_perf).sum()) / (1 + n_perm)
    return dict(
        n=hg.n, m=hg.m, kappa=hg.kappa, n_cross=hg.n_cross_total,
        copy_density=hg.n_cross_total / max(hg.m, 1),
        triple_fraction=hg.triple_fraction,
        loc_x=p_loc / pb, perfect_x=p_perf / pb, firstedge_x=p_first / pb,
        perfect_over_loc=s_perf,
        permuted_over_loc_mean=float(s_perm.mean()),
        permuted_over_loc_lo=float(np.percentile(s_perm, 2.5)),
        permuted_over_loc_hi=float(np.percentile(s_perm, 97.5)),
        mindeg_over_loc=p_mind / p_loc,
        mindeg_capture=(p_mind / p_loc - 1) / gain if abs(gain) > 1e-12 else np.nan,
        structural_share=float(np.mean(shares)),
        structural_share_lo=float(np.percentile(shares, 2.5)),
        structural_share_hi=float(np.percentile(shares, 97.5)),
        perm_pvalue=float(pval), n_perm=n_perm,
    )


def monte_carlo_check(hg, w, n_samples=200_000, seed=0):
    """Sample real paths from the weighted sampler; compare hit rate to the analytic p."""
    rng = np.random.default_rng(seed)
    wp = np.asarray(w, dtype=float) + 1.0
    pe = wp / wp.sum()
    D = hg._D(wp)
    picks = rng.choice(hg.m, size=n_samples, p=pe)
    hits = 0
    tset = {}
    for i in picks:
        u, v = int(hg.eu[i]), int(hg.ev[i])
        x, y = (u, v) if hg.rank[u] < hg.rank[v] else (v, u)
        a, b = hg.indptr[x], hg.indptr[x + 1]
        slots = hg.slot_eid[a:b]
        pr = wp[slots] / D[x]
        c = int(hg.indices[a + rng.choice(len(slots), p=pr / pr.sum())])
        if c == u or c == v or hg.rank[c] <= hg.rank[y]:
            continue
        key = (min(x, c), max(x, c))
        if key not in hg.eid or (min(y, c), max(y, c)) not in hg.eid:
            continue
        S = hg.memb[x] & hg.memb[y] & hg.memb[c]
        if not S:
            hits += 1
    return hits / n_samples


"""Additions for notebook 4: exact oracle-width on the projections, the cross-hyperedge
four-cycle pattern, and a diagnosis of why transfer fails on some graphs."""
import time
from collections import defaultdict

import numpy as np
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import maximum_flow


# ------------------------------------------------------- exact oracle-width (max-flow)

def _degeneracy_arrays(eu, ev):
    """Degeneracy of the graph given by parallel edge arrays. Returns (kappa, n_nodes)."""
    if len(eu) == 0:
        return 0, 0
    nodes = np.unique(np.concatenate([eu, ev]))
    remap = {int(v): i for i, v in enumerate(nodes)}
    n = len(nodes)
    adj = [[] for _ in range(n)]
    for a, b in zip(eu, ev):
        a, b = remap[int(a)], remap[int(b)]
        adj[a].append(b); adj[b].append(a)
    deg = np.array([len(a) for a in adj])
    maxd = int(deg.max())
    buckets = [set() for _ in range(maxd + 1)]
    for v in range(n):
        buckets[deg[v]].add(v)
    removed = np.zeros(n, dtype=bool)
    k, i = 0, 0
    for _ in range(n):
        while i <= maxd and not buckets[i]:
            i += 1
        if i > maxd:
            break
        v = buckets[i].pop()
        k = max(k, i)
        removed[v] = True
        for w in adj[v]:
            if removed[w]:
                continue
            d = int(deg[w])
            buckets[d].discard(w); deg[w] = d - 1; buckets[d - 1].add(w)
            if d - 1 < i:
                i = d - 1
    return k, n


def _orientation_feasible(eu, ev, k):
    m = len(eu)
    if m == 0:
        return True
    nodes = np.unique(np.concatenate([eu, ev]))
    pos = {int(v): i for i, v in enumerate(nodes)}
    n = len(nodes)
    S, E0, V0 = 0, 1, 1 + m
    T = 1 + m + n
    rows = np.empty(3 * m + n, dtype=np.int64)
    cols = np.empty(3 * m + n, dtype=np.int64)
    data = np.empty(3 * m + n, dtype=np.int32)
    idx = np.arange(m)
    rows[:m] = S;            cols[:m] = E0 + idx;  data[:m] = 1
    rows[m:2*m] = E0 + idx;  data[m:2*m] = 1
    cols[m:2*m] = V0 + np.array([pos[int(x)] for x in eu])
    rows[2*m:3*m] = E0 + idx; data[2*m:3*m] = 1
    cols[2*m:3*m] = V0 + np.array([pos[int(x)] for x in ev])
    rows[3*m:] = V0 + np.arange(n); cols[3*m:] = T; data[3*m:] = int(k)
    g = csr_matrix((data, (rows, cols)), shape=(T + 1, T + 1))
    return int(maximum_flow(g, S, T).flow_value) == m


def oracle_width(eu, ev, time_budget=None):
    """Exact pseudoarboricity of the given edge set; None if the budget runs out."""
    if len(eu) == 0:
        return 0
    hi, _ = _degeneracy_arrays(eu, ev)
    hi = max(hi, 1)
    lo = 1
    t0 = time.time()
    while lo < hi:
        if time_budget is not None and time.time() - t0 > time_budget:
            return None
        mid = (lo + hi) // 2
        if _orientation_feasible(eu, ev, mid):
            hi = mid
        else:
            lo = mid + 1
    return lo


def diagnostic_crossK3(hg, time_budget=900.0):
    """kappa, kappa_copy, alpha and the two ratios, for cross-triangles on a projection."""
    keep = hg.t > 0
    eu, ev = hg.eu[keep], hg.ev[keep]
    kappa_copy, _ = _degeneracy_arrays(eu, ev)
    t0 = time.time()
    alpha = oracle_width(eu, ev, time_budget=time_budget)
    return dict(
        m=hg.m, m_copy=int(keep.sum()), kappa=hg.kappa, kappa_copy=kappa_copy,
        alpha=alpha,
        alpha_lb=int(np.ceil(kappa_copy / 2)), alpha_ub=kappa_copy,
        alpha_over_kappa=(alpha / hg.kappa) if (alpha and hg.kappa) else None,
        kappa_copy_over_kappa=kappa_copy / hg.kappa if hg.kappa else None,
        n_cross=hg.n_cross_total, secs=round(time.time() - t0, 1),
    )


# ------------------------------------------------ cross-hyperedge four-cycles (rho = 2)

class CrossC4:
    """Cross-hyperedge four-cycles on the projection of a hypergraph.

    A 4-cycle a-x-b-y-a is *cross* when no single hyperedge contains all four vertices.
    rho(C4) = 2 and the Fichtenberger-Peng decomposition is a perfect matching of two
    edges, so BOTH elementary draws are base-edge draws against the global normalizer W.
    There is no closing-vertex step, hence no localization term: every gain measured here
    is attributable to the predictor alone.  That makes C4 the clean complement to K3.
    """

    def __init__(self, hg, max_c4=15_000_000, max_sumdeg2=5e7, max_n=3500, verbose=True):
        self.hg = hg
        self.ok = True
        self.reason = ''
        sumdeg2 = float((hg.deg.astype(float) ** 2).sum())
        self.n_c4_all = None
        if hg.n > max_n:
            self.ok = False
            self.reason = f'n = {hg.n} exceeds budget {max_n} (codegree table too large)'
            return
        if sumdeg2 > max_sumdeg2:
            self.ok = False
            self.reason = f'sum deg^2 = {sumdeg2:.2e} exceeds budget {max_sumdeg2:.1e}'
            return

        # adjacency lists from the CSR of hg
        nbr = [hg.indices[hg.indptr[v]:hg.indptr[v + 1]] for v in range(hg.n)]
        cod = defaultdict(list)
        for w in range(hg.n):
            a = np.sort(nbr[w])
            for i in range(len(a)):
                ai = int(a[i])
                for j in range(i + 1, len(a)):
                    cod[(ai, int(a[j]))].append(w)

        total = sum(len(v) * (len(v) - 1) // 2 for v in cod.values()) // 2
        self.n_c4_all = total
        if total > max_c4:
            self.ok = False
            self.reason = f'#C4 = {total:.3g} exceeds budget {max_c4:.3g}'
            return

        e1, e2 = [], []
        n_cross = 0
        eid = hg.eid
        memb = hg.memb
        for (a, b), L in cod.items():
            if len(L) < 2:
                continue
            L = sorted(L)
            mab = memb[a] & memb[b]
            for i in range(len(L)):
                x = L[i]
                for j in range(i + 1, len(L)):
                    y = L[j]
                    if (a, b) > (x, y):          # canonical: generate each C4 once
                        continue
                    if mab & memb[x] & memb[y]:  # covered by a single hyperedge
                        continue
                    n_cross += 1
                    ax = (a, x) if a < x else (x, a)
                    by = (b, y) if b < y else (y, b)
                    xb = (x, b) if x < b else (b, x)
                    ya = (y, a) if y < a else (a, y)
                    # canonical matching: the one holding the lexicographically least edge
                    if min(ax, by) <= min(xb, ya):
                        e1.append(eid[ax]); e2.append(eid[by])
                    else:
                        e1.append(eid[xb]); e2.append(eid[ya])
        self.e1 = np.array(e1, dtype=np.int32)
        self.e2 = np.array(e2, dtype=np.int32)
        self.n_cross = n_cross
        self.t = (np.bincount(self.e1, minlength=hg.m)
                  + np.bincount(self.e2, minlength=hg.m)).astype(np.int64)
        if verbose:
            print(f'    #C4(all)={self.n_c4_all}  #C4(cross)={self.n_cross}')

    # -- weightings
    def w_perfect(self):
        return self.t.astype(float)

    def w_uniform(self):
        return np.zeros(self.hg.m)

    def w_mindeg(self):
        return np.minimum(self.hg.deg[self.hg.eu], self.hg.deg[self.hg.ev]).astype(float)

    def w_permuted(self, w, seed):
        return np.random.default_rng(seed).permutation(np.asarray(w, dtype=float))

    def p_succ(self, w, first_edge_only=False):
        wp = np.asarray(w, dtype=float) + 1.0
        W = wp.sum()
        if first_edge_only:
            return float(wp[self.e1].sum()) / (W * self.hg.m)
        return float((wp[self.e1] * wp[self.e2]).sum()) / (W * W)

    def evaluate(self, n_perm=50, seed=0):
        p_u = self.p_succ(self.w_uniform())
        p_p = self.p_succ(self.w_perfect())
        p_m = self.p_succ(self.w_mindeg())
        p_f = self.p_succ(self.w_perfect(), first_edge_only=True)
        perms = np.array([self.p_succ(self.w_permuted(self.w_perfect(), seed + b))
                          for b in range(n_perm)])
        s_p, s_perm = p_p / p_u, perms / p_u
        gain = s_p - 1.0
        shares = (s_p - s_perm) / gain if abs(gain) > 1e-12 else np.full(n_perm, np.nan)
        return dict(
            m=self.hg.m, n_c4_all=self.n_c4_all, n_cross=self.n_cross,
            copy_density=self.n_cross / max(self.hg.m, 1),
            perfect_over_unif=s_p,
            permuted_over_unif_mean=float(s_perm.mean()),
            permuted_over_unif_lo=float(np.percentile(s_perm, 2.5)),
            permuted_over_unif_hi=float(np.percentile(s_perm, 97.5)),
            mindeg_over_unif=p_m / p_u,
            firstedge_over_unif=p_f / p_u,
            structural_share=float(np.mean(shares)),
            structural_share_lo=float(np.percentile(shares, 2.5)),
            structural_share_hi=float(np.percentile(shares, 97.5)),
            perm_pvalue=float((1 + (s_perm >= s_p).sum()) / (1 + n_perm)),
        )


# --------------------------------------------------------- why transfer fails somewhere

def feature_diagnosis(hg, label=''):
    """Signal available to a predictor on this graph: target spread and feature strength."""
    y = hg.y                       # log1p(t_cross) per edge
    t = hg.t
    out = dict(label=label, m=hg.m, n=hg.n,
               frac_zero=float((t == 0).mean()),
               t_mean=float(t.mean()), t_std=float(t.std()),
               t_cv=float(t.std() / t.mean()) if t.mean() else np.nan,
               y_std=float(y.std()),
               max_hyperedge=int(hg.hsize.max()),
               gini=float(_gini(t.astype(float))))
    for blk in ['A', 'AB', 'ABC']:
        X = hg.F[blk]
        cors = [abs(np.corrcoef(X[:, j], y)[0, 1]) if X[:, j].std() > 0 else 0.0
                for j in range(X.shape[1])]
        out[f'best_feat_corr_{blk}'] = float(np.nanmax(cors))
        out[f'R2_in_sample_{blk}'] = float(_r2(X, y))
    return out


def _gini(x):
    x = np.sort(x)
    n = len(x)
    if n == 0 or x.sum() == 0:
        return 0.0
    return float((2 * np.arange(1, n + 1) - n - 1).dot(x) / (n * x.sum()))


def _r2(X, y):
    Z = (X - X.mean(0)) / (X.std(0) + 1e-12)
    Z = np.hstack([Z, np.ones((len(Z), 1))])
    beta, *_ = np.linalg.lstsq(Z, y, rcond=None)
    resid = y - Z @ beta
    return 1 - resid.var() / (y.var() + 1e-12)


## 1. Data — the same ten hypergraphs

In [ ]:
!pip -q install gdown

import gdown, tarfile, glob, os, time, itertools
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option('display.width', 240)

ARB_IDS = {
    'contact-primary-school': '1sBHSEIyvVKavAho524Ro4cKL66W6rn-t',
    'contact-high-school':    '1VA2P62awVYgluOIh1W4NZQQgkQCBk-Eu',
    'email-Enron':            '1tTVZkdpgRW47WWmsrdUCukHz0x2M6N77',
    'email-Eu':               '1amLeVudLBDRglCXKlieg6HHE-vu81EVF',
    'NDC-classes':            '1tpDiP1c73O18gCYEx4OI7kx8V_IdYxLt',
    'NDC-substances':         '1mGOg0DMh46J2zQdimSXMde1pKNtfAdh8',
    'DAWN':                   '1wGwoG7oBWnNN7J9TEpjqNpODbsYfMxp4',
    'congress-bills':         '1gH1uJMZpn_SCJSRbORPH4JRQeevLwTyO',
    'tags-math-sx':           '1eDevpF6EZs19rLouNpiKGLIlFOLUfKKG',
    'tags-ask-ubuntu':        '1tb1ZJlXEJnlRkXpTuBZlOqqsFknWCkUV',
}
DOMAIN = {'contact-primary-school':'contact', 'contact-high-school':'contact',
          'email-Enron':'email', 'email-Eu':'email',
          'NDC-classes':'drugs', 'NDC-substances':'drugs', 'DAWN':'drugs',
          'congress-bills':'legislation',
          'tags-math-sx':'tags', 'tags-ask-ubuntu':'tags'}

def load_simplices(nv, sp):
    sizes = [int(x) for x in open(nv).read().split()]
    flat = [int(x) for x in open(sp).read().split()]
    out, i = [], 0
    for s in sizes:
        out.append(tuple(flat[i:i + s])); i += s
    return out

def _extract(tgz):
    with tarfile.open(tgz) as t:
        try:
            t.extractall('.', filter='data')
        except TypeError:
            t.extractall('.')

def fetch_arb(name, tries=3):
    tgz = f'{name}.tar.gz'
    for k in range(tries):
        try:
            if not os.path.exists(tgz) or os.path.getsize(tgz) < 1000:
                if not gdown.download(id=ARB_IDS[name], output=tgz, quiet=True):
                    gdown.download(url=f'https://drive.google.com/uc?id={ARB_IDS[name]}',
                                   output=tgz, quiet=True, fuzzy=True)
            _extract(tgz)
            nv = glob.glob(f'**/{name}-nverts.txt', recursive=True)
            sp = glob.glob(f'**/{name}-simplices.txt', recursive=True)
            return load_simplices(nv[0], sp[0])
        except Exception as e:
            print(f'  {name}: attempt {k+1}/{tries} ({type(e).__name__})')
            if os.path.exists(tgz) and os.path.getsize(tgz) < 1000:
                os.remove(tgz)
            time.sleep(5)
    return None

HYPER = {}
for name in ARB_IDS:
    h = fetch_arb(name)
    if h is None:
        continue
    uniq = list(dict.fromkeys(tuple(sorted(set(x))) for x in h))
    HYPER[name] = [x for x in uniq if 2 <= len(x) <= 25]
    print(f'{name:24s} {len(HYPER[name]):>7d} unique simplices')

MAX_TRIPLES = 4_000_000
HG = {}
for name, h in HYPER.items():
    print(f'{name} ...')
    HG[name] = CrossHG(h, max_size=25, max_triples=MAX_TRIPLES, seed=0)

## 2. Gap 1 — the diagnostic on all ten graphs

$\alpha_H$ exactly, by binary search over max-flow feasibility. `congress-bills` has
$4.2\times10^5$ copy-bearing edges, so the flow network has over a million arcs; a time
budget is set and, if it is exceeded, the bracket
$\lceil \kappa_H/2 \rceil \le \alpha_H \le \kappa_H$ from the lemma is reported instead —
which is itself enough to make the paper's point, since the lower end already exceeds
$0.5\,\kappa_H/\kappa$.

In [ ]:
TIME_BUDGET = 1200.0   # seconds per graph for the exact alpha

rows = []
for name, hg in HG.items():
    t0 = time.time()
    r = diagnostic_crossK3(hg, time_budget=TIME_BUDGET)
    r['label'], r['domain'] = name, DOMAIN[name]
    rows.append(r)
    a = f"{r['alpha']}" if r['alpha'] is not None else f"[{r['alpha_lb']},{r['alpha_ub']}]"
    ratio = (f"{r['alpha_over_kappa']:.3f}" if r['alpha_over_kappa'] is not None
             else f"[{r['alpha_lb']/r['kappa']:.3f},{r['alpha_ub']/r['kappa']:.3f}]")
    print(f"{name:24s} kappa={r['kappa']:>4d} kappa_copy={r['kappa_copy']:>4d} "
          f"alpha={a:>12s} alpha/kappa={ratio:>14s} "
          f"kappa_copy/kappa={r['kappa_copy_over_kappa']:.3f}  ({time.time()-t0:.0f}s)")

D1 = pd.DataFrame(rows)
D1[['label','domain','m','m_copy','kappa','kappa_copy','alpha_lb','alpha','alpha_ub',
    'alpha_over_kappa','kappa_copy_over_kappa','n_cross']].round(4)

In [ ]:
ok = D1.dropna(subset=['alpha_over_kappa'])
print(f'exact alpha on {len(ok)}/{len(D1)} graphs')
if len(ok):
    print(f'alpha/kappa      {ok.alpha_over_kappa.min():.3f} - {ok.alpha_over_kappa.max():.3f} '
          f'(mean {ok.alpha_over_kappa.mean():.3f})')
print(f'kappa_copy/kappa {D1.kappa_copy_over_kappa.min():.3f} - '
      f'{D1.kappa_copy_over_kappa.max():.3f} (mean {D1.kappa_copy_over_kappa.mean():.3f})')
bad = D1[(D1.alpha.notna()) & ((D1.alpha < D1.alpha_lb) | (D1.alpha > D1.alpha_ub))]
print('bracket violations:', len(bad))
print('\nPublished band for ordinary triangles on SNAP: [0.52, 0.89], mean 0.72')

## 3. Gap 2 — cross-hyperedge four-cycles

Enumeration is the binding cost: the number of four-cycles in a dense projection can be
astronomical, so the run is guarded by $\sum_v \deg(v)^2$, by $n$, and by the total $C_4$
count, and graphs over budget are skipped with the reason printed. This is a generality
check on a subset, not a second full sweep, and the manuscript should say so.

Note the reporting change. For $K_3$ there is a localization term and a prediction term;
for $C_4$ there is no closing-vertex draw, so everything below is measured against the
**unweighted** sampler and is attributable to the predictor alone.

In [ ]:
C4 = {}
rows = []
for name, hg in HG.items():
    print(f'{name} ...')
    t0 = time.time()
    c = CrossC4(hg, max_c4=15_000_000, max_sumdeg2=5e7, max_n=3500)
    if not c.ok:
        print(f'    skipped: {c.reason}')
        continue
    C4[name] = c
    r = c.evaluate(n_perm=50, seed=0)
    r['label'], r['domain'], r['secs'] = name, DOMAIN[name], round(time.time() - t0, 1)
    rows.append(r)
    print(f"    pred={r['perfect_over_unif']:.3f}x  perm={r['permuted_over_unif_mean']:.3f}x  "
          f"share={r['structural_share']:.3f}  p={r['perm_pvalue']:.4f}  ({r['secs']}s)")

D2 = pd.DataFrame(rows)
D2[['label','domain','m','n_c4_all','n_cross','copy_density','perfect_over_unif',
    'permuted_over_unif_mean','permuted_over_unif_lo','permuted_over_unif_hi',
    'structural_share','structural_share_lo','structural_share_hi','perm_pvalue',
    'mindeg_over_unif','firstedge_over_unif']].round(4) if len(D2) else 'all graphs skipped'

In [ ]:
if len(D2):
    print('=== does the permutation reversal hold for rho(H) = 2 ? ===')
    print(f'perfect/unif   {D2.perfect_over_unif.min():.3f} - {D2.perfect_over_unif.max():.3f}')
    print(f'permuted/unif  {D2.permuted_over_unif_mean.min():.3f} - '
          f'{D2.permuted_over_unif_mean.max():.3f}   '
          f'(below 1 on {int((D2.permuted_over_unif_mean < 1).sum())}/{len(D2)} graphs)')
    print(f'struct share   {D2.structural_share.min():.3f} - {D2.structural_share.max():.3f} '
          f'(mean {D2.structural_share.mean():.3f})')
    print(f'max p-value    {D2.perm_pvalue.max():.4f}')
    print()
    print('For K3 the same quantities were: share 1.008-1.153, permuted 0.931-0.999.')
    print('For ordinary triangles (Array 2026): structural share 0.093.')

If the structural share is again near $1$ here, the claim in the manuscript can be
stated for higher-order patterns in general rather than for triangles only, and the
abstract's sentence about the reversal gains a second pattern. If it drops towards the
$0.09$ of ordinary triangles, the claim must be narrowed to $K_3^{\times}$ — and that
would itself be interesting, since it would tie the effect to the closing-vertex draw.

## 4. Gap 3 — why transfer fails on `contact-high-school`

The working explanation in the draft: with $327$ vertices and simplices of size at most
$5$, the per-edge cross-triangle count barely varies, so there is almost nothing for a
model to predict. Two ways to check it. First, the spread of the target itself. Second,
whether the same statistic explains the *whole* spread of transfer quality across the ten
graphs, which would turn an excuse for one row into a finding.

In [ ]:
rowsF = [feature_diagnosis(hg, name) for name, hg in HG.items()]
F = pd.DataFrame(rowsF)
F[['label','n','m','max_hyperedge','frac_zero','t_mean','t_std','t_cv','y_std','gini',
   'best_feat_corr_A','best_feat_corr_ABC','R2_in_sample_A','R2_in_sample_ABC']].round(3)

In [ ]:
# paste the per-graph capture numbers from notebook 3 (table2_pairs_raw.csv),
# or recompute them here if that file is at hand.
CAPTURE = {   # block ABC, lambda 0.1, mean over the 9 training graphs
    'DAWN': 0.219, 'NDC-classes': 0.310, 'NDC-substances': 0.714,
    'congress-bills': 0.624, 'contact-high-school': -0.076,
    'contact-primary-school': 0.308, 'email-Enron': 0.270, 'email-Eu': 0.618,
    'tags-ask-ubuntu': 0.887, 'tags-math-sx': 0.825,
}
try:
    raw = pd.read_csv('table2_pairs_raw.csv')
    d = raw[(raw.block == 'ABC') & (raw.lam == 0.1)]
    CAPTURE = d.groupby('test')['capture'].mean().to_dict()
    print('capture read from table2_pairs_raw.csv')
except Exception:
    print('using the hard-coded capture values from notebook 3')

F['capture'] = F.label.map(CAPTURE)
G = F.dropna(subset=['capture'])
for col in ['y_std', 't_cv', 'gini', 'best_feat_corr_ABC', 'R2_in_sample_ABC',
            'max_hyperedge', 'n', 'm']:
    r = np.corrcoef(G[col], G.capture)[0, 1]
    print(f'corr(capture, {col:>18s}) = {r:+.3f}')

chs = F[F.label == 'contact-high-school']
print('\ncontact-high-school vs the rest:')
print(F[['label','y_std','t_cv','gini','best_feat_corr_ABC','R2_in_sample_ABC','capture']]
      .round(3).sort_values('capture').to_string(index=False))

In [ ]:
if len(G) > 2:
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
    ax[0].scatter(G.y_std, G.capture)
    for _, r in G.iterrows():
        ax[0].annotate(r.label[:12], (r.y_std, r.capture), fontsize=6)
    ax[0].set_xlabel('std of log(1+t(e))  — spread of the target')
    ax[0].set_ylabel('leave-one-graph-out capture'); ax[0].axhline(0, lw=.5, c='k')
    ax[1].scatter(G.R2_in_sample_ABC, G.capture, c='crimson')
    for _, r in G.iterrows():
        ax[1].annotate(r.label[:12], (r.R2_in_sample_ABC, r.capture), fontsize=6)
    ax[1].set_xlabel('in-sample $R^2$ of the feature block')
    ax[1].set_ylabel('capture'); ax[1].axhline(0, lw=.5, c='k')
    plt.tight_layout(); plt.show()

## 5. Save, and print the manuscript rows

In [ ]:
D1.to_csv('gap1_diagnostic_all10.csv', index=False)
if len(D2):
    D2.to_csv('gap2_crossC4.csv', index=False)
F.to_csv('gap3_feature_diagnosis.csv', index=False)
print('saved')

print('\n%%%% ---- Table: diagnostic on all ten graphs ----')
for _, r in D1.iterrows():
    a = f"{int(r.alpha)}" if pd.notna(r.alpha) else f"[{int(r.alpha_lb)},{int(r.alpha_ub)}]"
    ratio = (f"{r.alpha_over_kappa:.3f}" if pd.notna(r.alpha_over_kappa)
             else f"[{r.alpha_lb/r.kappa:.2f},{r.alpha_ub/r.kappa:.2f}]")
    print(f"{r.label} & {r.domain} & {int(r.m):,} & {int(r.kappa)} & {int(r.kappa_copy)} & "
          f"{a} & {ratio} & {r.kappa_copy_over_kappa:.3f} \\\\".replace(',', '{,}'))

if len(D2):
    print('\n%%%% ---- Table: cross-hyperedge four-cycles ----')
    for _, r in D2.iterrows():
        print(f"{r.label} & {r.domain} & {int(r.m):,} & {int(r.n_cross):,} & "
              f"{r.copy_density:.1f} & {r.perfect_over_unif:.2f} & "
              f"{r.permuted_over_unif_mean:.3f} & {r.structural_share:.3f} & "
              f"{r.mindeg_over_unif:.2f} \\\\".replace(',', '{,}'))

## 6. What to change in the manuscript

* **Gap 1** — replace Table 1 with the ten-graph version printed above, and delete the
  `\GAP` box in Section 6.2. If the exact $\alpha$ timed out on `congress-bills`, report the
  bracket for that row and add one sentence: the lower end of the bracket already exceeds
  the separation threshold, so the conclusion does not depend on the exact value.
* **Gap 2** — add the $C_4$ table as a subsection of Section 6, and rewrite the sentence in
  Section 6.8. State plainly that the $C_4$ sweep covers a subset of the datasets and why
  (enumeration cost), and state the structural finding that $C_4$ carries no localization
  term, which is why its numbers are reported against the unweighted sampler.
* **Gap 3** — replace the `\GAP` box in Section 6.5 with the measured explanation. If
  `capture` correlates with the target spread across all ten graphs, promote it from an
  excuse into a stated regularity: a predictor helps only where the per-edge weight varies
  enough to be worth predicting, and that is checkable in one pass before training.